In [ ]:
import os
import glob
import numpy as np
import xarray as xr

import matplotlib
from matplotlib import cm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1 import make_axes_locatable
%matplotlib inline

from joblib import Parallel, delayed

# --- gridded NetCDF + per-basin river profiles (PyGMT/ArcGIS + matplotlib) ---
from gospl.analyse.gridexport import (
    grid_export, to_netcdf, basin_rivers, plot_long_profile, plot_basin_map)

# --- stratigraphic sections / wells / Wheeler (matplotlib, inline) ---
from gospl.analyse.stratasection import (
    load_strata, cross_section, horizontal_slice, synthetic_well, wheeler,
    well_panel)

## Running the simulation

First activate the conda environment:

```bash
conda activate gospl
```

To run the simulation in a terminal (`X` = number of MPI processes, e.g. 5):

```bash
mpirun -np X gospl -i input-500.yml
```

# Analysing the outputs

All the post-processing below uses goSPL's built-in **`gospl.analyse`** toolkit
(imported in the first cell). Two complementary modules are used:

- **`gospl.analyse.gridexport`** — reassembles the unstructured mesh, rasterises
  every surface field of an output step onto a **regular grid**, runs D8
  hydrology (drainage area, basins, &chi;) and writes a CF-NetCDF for
  PyGMT/ArcGIS. It also exposes per-basin **river long-profile** helpers.
- **`gospl.analyse.stratasection`** — reads the recorded stratigraphy and draws
  **cross-sections, synthetic wells and Wheeler (chronostratigraphic)
  diagrams** (coloured by facies, lithology, provenance, &hellip;).

Every function used below has a **terminal equivalent** so the same products can
be generated outside Jupyter. These console commands &mdash; `gospl-grid`,
`gospl-section`, `gospl-strata-volume` &mdash; are installed with goSPL and are
shown in each section.

Each gridded NetCDF (one file per output step) holds, when available (every
variable carries its `units` and a `long_name` definition):

+ surface elevation `elev` (m) and the step's `sea_level` (m)
+ cumulative erosion/deposition `erodep` (m) and its rate `EDrate` (m/yr)
+ water / sediment fluxes `FA`, `fillFA`, `waterFill`, `sedLoad`
+ hydrology: `drainage_area`, `basin` id, `chi`, `flowdist`, and the
  priority-flood-`filled` elevation

In [ ]:
# Define output folder name for the simulation
out_path = 'results/'

if not os.path.exists(out_path):
    os.makedirs(out_path)

### Rasterising the outputs to a regular grid &mdash; `grid_export` / `to_netcdf`

`grid_export` reassembles the global mesh, interpolates a step's fields onto a
regular grid, runs the D8 hydrology and returns a dict of 2-D arrays;
`to_netcdf` writes that to a CF-NetCDF (each variable annotated with its `units`
and `long_name`). `getOutputs` below simply loops over the steps and writes one
`results/surface<step>.nc` per step.

**`grid_export(h5dir, mesh, step=None, ...)` &mdash; main options**

| Argument | Default | Meaning |
|---|---|---|
| `h5dir` | &ndash; | the run's `h5` output directory |
| `mesh` | &ndash; | global mesh `.npz` (vertices `v`, cells `c`) |
| `step` | last | output step to rasterise |
| `spacing` | median edge | grid resolution `dx[,dy]` (mesh units) |
| `fields` | all | subset of surface fields to include |
| `mn` | `0.5` | &chi; concavity `m/n` |
| `a0` | `1.0` | &chi; reference drainage area |
| `base_level` | run sea level | elevation defining the coast / outlets (catchment + &chi; datum) |
| `latlim` | `89` | (global meshes) crop the polar caps |

Global (spherical) meshes are auto-detected and gridded in lon/lat. The resolved
sea level is stored in each file (global attribute **and** a `sea_level`
variable), so the grid is self-describing.

**Terminal equivalent** (one step &rarr; one NetCDF):

```bash
gospl-grid --h5dir sim_slope_500/h5 --mesh data/gospl_mesh.npz:v:c \
    --step 40 --spacing 200 --out results/500_surface_40.nc
```

For the whole time series, loop in the shell:

```bash
for s in $(seq 0 11); do
  gospl-grid --h5dir sim_slope_500/h5 --mesh data/gospl_mesh.npz:v:c \
      --step $s --spacing 200 --out results/500_surface_$s.nc
done
```

In [ ]:
h5dir = "sim_slope_500/h5"
mesh = "data/gospl_mesh.npz"
reso = 200

out_name = "500_surface_"

def getOutputs(steps):

    # clear any stale .nc files first
    for f in glob.glob(os.path.join(out_path, f"{out_name}*.nc")):
        try:
            os.remove(f)
        except PermissionError:
            print(f"Still locked, close it first: {f}")
            return
        
    for stp in steps:
        g = grid_export(h5dir, mesh, stp, spacing=reso)
        fname = os.path.join(out_path, f"{out_name}{stp}.nc")
        to_netcdf(g, fname)                       

    return

def getOutputsParallel(steps, h5dir, n_workers=8):
    for f in glob.glob(os.path.join(out_path, f"{out_name}*.nc")):
        try:
            os.remove(f)
        except PermissionError:
            print(f"Still locked, close it first: {f}")
            return

    def process_step(stp):
        g = grid_export(h5dir, mesh, stp, spacing=reso)
        fname = os.path.join(out_path, f"{out_name}{stp}.nc")
        to_netcdf(g, fname)

    Parallel(n_jobs=n_workers)(delayed(process_step)(stp) for stp in steps)

steps = np.arange(41)
getOutputsParallel(steps, h5dir, n_workers=8)
# getOutputs(steps)

In [ ]:
h5dir = "sim_slope_1.25k/h5"
out_name = "125_surface_"
getOutputsParallel(steps, h5dir, n_workers=8)

In [ ]:
h5dir = "sim_slope_2.5k/h5"
out_name = "250_surface_"
getOutputsParallel(steps, h5dir, n_workers=8)

In [ ]:
h5dir = "sim_slope_5k/h5"
out_name = "5k_surface_"
steps = np.arange(21)
getOutputsParallel(steps, h5dir, n_workers=8)

### Comparing the final topographies

The four panels show the steady-state elevation field for each $\Delta t$ at the end of the run. Look for whether the large-scale relief and drainage organisation are preserved across time steps: the implicit scheme keeps every solution stable and broadly similar, but the detailed valley/divide positions drift as $\Delta t$ grows, because larger steps smear out transient features such as migrating knickpoints.

### How consistent is the maximum elevation?

A compact accuracy check: the peak elevation reached by each run should be nearly identical if $\Delta t$ has little effect on the steady state. The mean and standard deviation across the four runs quantify the spread; a small relative deviation confirms that even the coarse 5 ky step recovers the same overall relief amplitude.

In [ ]:
sim1 = xr.open_dataset("results/500_surface_40.nc")
sim2 = xr.open_dataset("results/125_surface_40.nc")
sim3 = xr.open_dataset("results/250_surface_40.nc")
sim4 = xr.open_dataset("results/5k_surface_20.nc")

### Mean-elevation history: accuracy vs $\Delta t$

This plots the domain-averaged elevation through time for each $\Delta t$. All curves climb and then plateau, showing every run reaches steady state regardless of step size (the payoff of the unconditionally stable implicit scheme). Watch how closely the curves overlap: divergence between them grows with $\Delta t$, so the gap quantifies the accuracy cost of taking larger steps to gain speed. The detailed interpretation follows below.

In [ ]:
fig, axs = plt.subplots(2,2, figsize=(8,8), sharex=True, sharey=True)
sim1.elev.plot(ax=axs[0,0], add_labels=False, add_colorbar=False)
sim2.elev.plot(ax=axs[0,1], add_labels=False, add_colorbar=False)
sim3.elev.plot(ax=axs[1,0], add_labels=False, add_colorbar=False)
im = sim4.elev.plot(ax=axs[1,1], add_labels=False, add_colorbar=False)

axs[0,0].set_title('dt = 500 yrs', fontsize=10, fontweight="bold")
axs[0,1].set_title('dt = 1,250 yrs', fontsize=10, fontweight="bold")
axs[1,0].set_title('dt = 2,500 yrs', fontsize=10, fontweight="bold")
axs[1,1].set_title('dt = 5,000 yrs', fontsize=10, fontweight="bold")

cbar_ax = fig.add_axes([0.2, -0.02, 0.6, 0.02]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Elevation (m)')

# plt.tight_layout()
plt.show()

Run the following data preparation or mesh generation step.

In [ ]:
maxelev = np.zeros(4)
maxelev[0] = sim1.elev.max().data+0.
maxelev[1] = sim2.elev.max().data+0.
maxelev[2] = sim3.elev.max().data+0.
maxelev[3] = sim4.elev.max().data+0.

print('Average max. elevation between the different simulation: %0.2f m'%maxelev.mean())
print('Standard deviation: %0.2f m'%maxelev.std())

Extracting mean elevation through time:

In [ ]:
out_path = 'results/'
def get_mean_elevation(ncout, maxtsp):
    mean_z = []
    for k in range(maxtsp+1):
        ds = xr.open_dataset(ncout+str(k)+".nc")
        mean_z.append(ds.elev.mean().data+0.)
    return mean_z

mean1 = get_mean_elevation(os.path.join(out_path, "500_surface_"), 40)
mean2 = get_mean_elevation(os.path.join(out_path, "125_surface_"), 40)
mean3 = get_mean_elevation(os.path.join(out_path, "250_surface_"), 40)
mean4 = get_mean_elevation(os.path.join(out_path, "5k_surface_"), 20) 

Create coordinate or time arrays used in the analysis.

In [ ]:
time1 = np.arange(41)*2.5e3
time2 = np.arange(21)*5.e3

fig, ax = plt.subplots(1,1, figsize=(4.5,6))
plt.plot(time1/1000, mean1, label='dt = 0.5 ky', marker='^', lw=2)
plt.plot(time1/1000, mean2, label='dt = 1.25 ky', marker='X', lw=2)
plt.plot(time1/1000, mean3, label='dt = 2.5 ky', marker='o', lw=2)
plt.plot(time2/1000, mean4, label='dt = 5 ky', marker='v', lw=2)

ax.legend(loc=4, frameon=False, fontsize=8, prop={'weight':'bold'})
plt.xlabel("Simulation time (ky)", fontsize=10, fontweight="bold")
plt.ylabel("Mean elevation (m)", fontsize=10, fontweight="bold")

plt.axis([-2, 102, 95, 185])

plt.tight_layout()
plt.show()

**The solutions for the mean landscape elevation show that the landscape reaches steady state in all cases, and overall the final elevations are in good agreement with a maximum elevation of 463±13 m.** 

Yet as the time step increases the differences between models increase over time. By the end of the simulation, the mean elevation difference between the case with Δt equals 500 years and the one at 5,000 years is around 6.5 %, whereas the difference with a Δt of 1250 years is below 1 %. It illustrates the transient nature of the landscape and its strong dependence on antecedent morphologies. Even small changes in elevation could potentially trigger completely different landscape features. 

> Compared to the explicit algorithm, the approach here relies on an implicit schema and produces a more stable solution for longer timescales. Yet time step limitations are still required to ensure a good representation of landscape features (e.g. knickpoint propagation) and care should be taken when choosing a given simulation time step.

The iterative linear solvers of the implicit methods for both flow accumulation and erosion use previous time step solution as an initial guess. In cases in which the landscape does not change significantly between consecutive time steps, both the flow accumulation and erosion rates are likely to remain almost unchanged and the number of iterations required by the solver to reach convergence will be small. As an example, if the drainage network remains the same between two iterations, the flow accumulation solver solution will be obtained immediately and the results given directly. It highlights a second implication of the choice of time step. Not only does the time step influence the final landscape morphology, but it also controls the model running time. In some cases, similar running times will be achieved with smaller time steps if solver solutions are obtained in a reduced number of iterations.